# Phase 2–3: verification, extended analysis, and foundation-model baseline

**Do not run this until `CXR_Ensemble_Experiments.ipynb` has finished all of its cells.** Use a fresh runtime.

This notebook never modifies `runs/` or `results/`. It only reads them, and writes to:
- `runs_v3/` (new training runs: EVA-X, extra seeds)
- `results_v3/` (extended analysis, tables, figures)
- `verify_report.json`

Put the `v3` folder (with `analysis_v3.py`, `verify_run.py`, `train_v3.py`, `evax.py`) inside `Colab Notebooks/cxr_experiment/`.

| Cell | Needs GPU | Time (A100, estimated) |
|---|---|---|
| 1–3 setup, data, verification | no | ~5 min |
| 4 extended analysis of the finished run | no | ~10 min |
| 5 EVA-X-S fine-tuning, two recipes | yes | ~1–2 h |
| 6 per-model inference benchmark | yes | ~10 min |
| 7 analysis with EVA-X | no | ~15 min |
| 8 optional: EVA-X-B, extra seeds | yes | 1–6 h |

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install timm albumentations kagglehub huggingface_hub

import sys, os, glob, json, importlib
hits = glob.glob('/content/drive/MyDrive/Colab Notebooks/cxr_experiment/cxr_pipeline.py') or \
       glob.glob('/content/drive/MyDrive/**/cxr_experiment/cxr_pipeline.py', recursive=True)
PROJECT = os.path.dirname(hits[0])
V3 = f'{PROJECT}/v3'
assert os.path.exists(f'{V3}/analysis_v3.py'), 'Upload the v3 folder into cxr_experiment first'
sys.path[:0] = [V3, PROJECT]
import cxr_pipeline as cx, analysis_v3, verify_run, train_v3, evax
for m in (cx, analysis_v3, verify_run, train_v3, evax): importlib.reload(m)
LOCAL_CACHE = '/content/cache'
print('Project:', PROJECT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Data (copies the cache from Drive; no download)

In [ ]:
cx.get_data(LOCAL_CACHE, f'{PROJECT}/cache')
os.makedirs(f'{PROJECT}/results_v3', exist_ok=True)
!pip freeze > "{PROJECT}/results_v3/environment_pip_freeze.txt"
!nvidia-smi > "{PROJECT}/results_v3/environment_gpu.txt"

## 3. Integrity check of the finished run
Read-only. **If anything says FAIL, stop and send the output before interpreting any numbers.**

In [ ]:
!python "{V3}/verify_run.py" --project "{PROJECT}" --cache {LOCAL_CACHE} --report "{PROJECT}/verify_report.json"

## 4. Extended analysis of the five-model run (CPU)
Fusion methods (averaging, calibrated averaging, validation-weighted averaging, label-wise and full stacking),
AUROC/AUPRC/F1/ECE/Brier, best subset of each size (chosen on validation), error-correlation diversity,
per-finding gains with Holm correction, subgroup CIs. Everything fitted on validation only.

In [ ]:
R5 = analysis_v3.analyze(f'{PROJECT}/runs', LOCAL_CACHE, f'{PROJECT}/results_v3/base5', n_boot=1000)

## 5. EVA-X-S baseline (GPU)
Fine-tunes EVA-X-S from its public chest X-ray self-supervised weights on **our** training split, selecting the
checkpoint on **our** validation split. Two recipes: `shared` (identical to the five backbones) and `evax`
(EVA-X's own optimizer settings). Resumable. Writes only to `runs_v3/`.

In [ ]:
for recipe in ['shared', 'evax']:
    train_v3.train(PROJECT, LOCAL_CACHE, 'evax_s', recipe=recipe, weights_dir='/content/hf')

## 5b. ViT-B/16 stability rerun (GPU, ~0.5 h) — sensitivity analysis
In the main run, ViT-B/16 diverged at epoch 8 (loss rose, validation AUROC fell from 0.74 to 0.56) under the shared
learning rate of 1e-4. This reruns it with 3e-5, everything else identical, into `runs_v3/vit_lr3e-5/`.
The main analysis keeps the pre-specified run; this one is reported separately as a disclosed sensitivity check.

In [ ]:
train_v3.train(PROJECT, LOCAL_CACHE, 'vit_b16', recipe='shared', overrides={'lr': 3e-5}, tag='vit_lr3e-5')

## 6. Per-model inference benchmark (GPU, ~10 min)

In [ ]:
bench = train_v3.benchmark_models(PROJECT, LOCAL_CACHE, f'{PROJECT}/results_v3/bench_v3.json')

## 7. Analysis with the foundation model
Adds EVA-X-S (shared recipe, and separately its own recipe) to the pool and runs the pre-specified comparisons:
ImageNet five-model stacking vs EVA-X; stacking with vs without EVA-X; EVA-X alone vs the ensemble.

In [ ]:
for recipe_dir, tag in [('main', 'fm_shared'), ('evaxrecipe', 'fm_evaxrecipe')]:
    p = f'{PROJECT}/runs_v3/{recipe_dir}/evax_s/preds.npz'
    if os.path.exists(p):
        analysis_v3.analyze(f'{PROJECT}/runs', LOCAL_CACHE, f'{PROJECT}/results_v3/{tag}', n_boot=1000,
                            bench_json=f'{PROJECT}/results_v3/bench_v3.json', extra_sources=[('evax_s', p)])

In [ ]:
# Sensitivity: same analysis with the stable ViT-B/16 rerun swapped in
p_vit = f'{PROJECT}/runs_v3/vit_lr3e-5/vit_b16/preds.npz'
p_fm = f'{PROJECT}/runs_v3/main/evax_s/preds.npz'
if os.path.exists(p_vit):
    extra = [('evax_s', p_fm)] if os.path.exists(p_fm) else []
    analysis_v3.analyze(f'{PROJECT}/runs', LOCAL_CACHE, f'{PROJECT}/results_v3/sens_vit_lr3e-5', n_boot=1000,
                        bench_json=f'{PROJECT}/results_v3/bench_v3.json', extra_sources=extra,
                        replace_sources={'vit_b16': p_vit})

## 8. Optional, only after reviewing cell 7
- EVA-X-B (`'evax_b'`, ~1 h): stronger foundation baseline.
- Extra seeds to measure training variability, e.g. EVA-X-S and DenseNet121 with seeds 1 and 2 (~0.5 h and ~1.5 h each).

In [ ]:
RUN_OPTIONAL = False
if RUN_OPTIONAL:
    train_v3.train(PROJECT, LOCAL_CACHE, 'evax_b', recipe='evax', weights_dir='/content/hf')
    for seed in [1, 2]:
        train_v3.train(PROJECT, LOCAL_CACHE, 'evax_s', recipe='shared', seed=seed, weights_dir='/content/hf')
        train_v3.train(PROJECT, LOCAL_CACHE, 'densenet121', recipe='shared', seed=seed)

## 9. Package Phase 2 outputs (send `results_v3.zip` back)

In [ ]:
import shutil
shutil.copy(f'{PROJECT}/verify_report.json', f'{PROJECT}/results_v3/verify_report.json')
print(shutil.make_archive(f'{PROJECT}/results_v3', 'zip', f'{PROJECT}/results_v3'))